In [15]:
import pandas as pd
import joblib
import os
from dotenv import load_dotenv
from conformist import FNRCoP, PredictionDataset, ValidationTrial

load_dotenv()

RANDOM_SEED = int(os.getenv("RANDOM_SEED", 42))
N_JOBS = int(os.getenv("N_JOBS", 10))
X_CALIBRATION_CSV_PATH = os.getenv("X_CALIBRATION_CSV_PATH")
Y_CALIBRATION_TSV_PATH = os.getenv("Y_CALIBRATION_TSV_PATH")
X_TEST_CSV_PATH = os.getenv("X_TEST_CSV_PATH")
Y_TEST_TSV_PATH = os.getenv("Y_TEST_TSV_PATH")
RF_MODEL_PATH = os.getenv("RF_MODEL_PATH")
SELECTED_FEATURES_PATH = os.getenv("SELECTED_FEATURES_PATH")
PREDS_CAL_CSV_PATH = os.getenv("PREDS_CAL_CSV_PATH")
PREDS_TEST_CSV_PATH = os.getenv("PREDS_TEST_CSV_PATH")
OUTPUT_DIR = os.getenv("OUTPUT_DIR")

# Print config
print(f"RANDOM_SEED: {RANDOM_SEED}")
print(f"N_JOBS: {N_JOBS}")
print(f"X_CALIBRATION_CSV_PATH: {X_CALIBRATION_CSV_PATH}")
print(f"Y_CALIBRATION_TSV_PATH: {Y_CALIBRATION_TSV_PATH}")
print(f"X_TEST_CSV_PATH: {X_TEST_CSV_PATH}")
print(f"Y_TEST_TSV_PATH: {Y_TEST_TSV_PATH}")
print(f"RF_MODEL_PATH: {RF_MODEL_PATH}")
print(f"SELECTED_FEATURES_PATH: {SELECTED_FEATURES_PATH}")
print(f"PREDS_CAL_CSV_PATH: {PREDS_CAL_CSV_PATH}")
print(f"PREDS_TEST_CSV_PATH: {PREDS_TEST_CSV_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

RANDOM_SEED: 42
N_JOBS: 10
X_CALIBRATION_CSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
Y_CALIBRATION_TSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
X_TEST_CSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-test/transpose/transposed.csv
Y_TEST_TSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-test/class/subtypes.tsv
RF_MODEL_PATH: /home/mariya/Data/pallas/training/final_model.pkl
SELECTED_FEATURES_PATH: /home/mariya/Data/pallas/training/selected_features.pkl
PREDS_CAL_CSV_PATH: /home/mariya/Data/pallas/conformal_prediction/gex_calibration_probs.csv
PREDS_TEST_CSV_PATH: /home/mariya/Data/pallas/conformal_prediction/gex_test_probs.csv
OUTPUT_DIR: /home/mariya/Data/pallas/conformal_prediction


In [16]:
# Load the model
rf = joblib.load(RF_MODEL_PATH)

# Load selected features
selected_features = joblib.load(SELECTED_FEATURES_PATH)

In [17]:
# Load calibration data
X_calibration = pd.read_csv(X_CALIBRATION_CSV_PATH, index_col=0)
y_calibration = pd.read_csv(Y_CALIBRATION_TSV_PATH, index_col=0, sep="\t").squeeze()
X_calibration = X_calibration[selected_features]
X_calibration = X_calibration.loc[y_calibration.index]
X_calibration.shape, y_calibration.shape

((459, 3931), (459,))

In [18]:
# Load test data
X_test = pd.read_csv(X_TEST_CSV_PATH, index_col=0)
y_test = pd.read_csv(Y_TEST_TSV_PATH, index_col=0, sep="\t").squeeze()
X_test = X_test[selected_features]
X_test = X_test.loc[y_test.index]
X_test.shape, y_test.shape

((513, 3931), (513,))

In [19]:
def format_probabilities_for_conformist(model, X, y, path):
    probs = model.predict_proba(X)
    preds = model.predict(X)

    df = pd.DataFrame(index=X.index)

    # Add a column with known_class
    df["known_class"] = y

    # Add a column with the predicted class
    df["predicted_class"] = preds

    probs_df = pd.DataFrame(probs, columns=model.classes_, index=X.index)
    df = pd.concat([df, probs_df], axis=1)

    # Rename index to "id"
    df = df.rename_axis("id")

    # Save
    df.to_csv(path)

In [20]:
format_probabilities_for_conformist(rf, X_calibration, y_calibration, PREDS_CAL_CSV_PATH)
format_probabilities_for_conformist(rf, X_test, y_test, PREDS_TEST_CSV_PATH)